# Historical Kaggle Measured Workflow

This sanitized notebook records the workflow used for the measured T4 experiment, adapter publication, rank pilot, and PubMedQA transfer check. It is retained for provenance; `cloud_runner.ipynb` remains the canonical training entry point.

The workflow checks out the exact source commit recorded by `clean_main_v1`. Hugging Face publication requires an authorized Kaggle `HF_TOKEN` secret. Git credential, commit, and push operations are intentionally excluded. This project is not clinically validated.


## Pinned environment

Clone the repository, check out the exact measured source revision, install dependencies, and verify the GPU.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/Leng-Bu-Ding/medical-llm-qlora.git"
SOURCE_COMMIT = "6a031506df4e1ad96765d67959f4aadeac01c54b"
PROJECT_DIR = Path("/kaggle/working/medical-llm-qlora")

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

subprocess.run(
    ["git", "checkout", "--detach", SOURCE_COMMIT],
    cwd=PROJECT_DIR,
    check=True,
)
os.chdir(PROJECT_DIR)
print("Using source commit:", SOURCE_COMMIT)


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements-train.txt"],
    check=True,
)


In [ ]:
subprocess.run(["nvidia-smi"], check=True)

import torch

assert torch.cuda.is_available(), "Select a Kaggle GPU accelerator first."
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(index, torch.cuda.get_device_name(index))


## Measured clean run

Run the canonical pipeline, verify its artifacts, inspect summaries, and create a downloadable runtime archive.


In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/run_pipeline.py",
        "--run-id",
        "clean_main_v1",
        "--protocol",
        "clean",
    ],
    check=True,
)


In [ ]:
from pathlib import Path

root = Path("/kaggle/working/medical-llm-qlora/outputs/clean_main_v1")

checks = {
    "Main Run 完成": root / "pipeline_manifest.json",
    "Adapter 权重": root / "training/adapter",
    "训练结果": root / "training/training_summary.json",
    "300题预测": root / "predictions_300.jsonl",
    "评测结果": root / "evaluation_summary.json",
    "Safety结果": root / "safety_summary.json",
}

for name, path in checks.items():
    print(f"{name}: {path.exists()}  ->  {path}")

In [ ]:
import json
from pathlib import Path

root = Path("/kaggle/working/medical-llm-qlora/outputs/clean_main_v1")

for name in [
    "training/training_summary.json",
    "evaluation_summary.json",
    "safety_summary.json",
    "pipeline_manifest.json",
]:
    path = root / name
    print("\n" + "=" * 20, name, "=" * 20)
    with open(path, encoding="utf-8") as f:
        print(json.dumps(json.load(f), indent=2, ensure_ascii=False))

In [ ]:
import shutil

src = "/kaggle/working/medical-llm-qlora/outputs/clean_main_v1"
dst = "/kaggle/working/clean_main_v1"

archive = shutil.make_archive(dst, "zip", src)
print(archive)

In [ ]:
from IPython.display import FileLink

FileLink("/kaggle/working/clean_main_v1.zip")

## Adapter publication and integrity checks

These owner-side cells use a Kaggle `HF_TOKEN` secret to publish or restore the private adapter and compare restored behavior. Review the target repository before running the upload cell.


In [ ]:
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("HF_TOKEN")
print("HF_TOKEN loaded:", token is not None)

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi

token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token=token)

repo_id = "Lengbuding/llama3-medquad-qlora"

# 创建模型仓库（先设为 private）
api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    private=True,
    exist_ok=True,
)

# 上传训练好的 QLoRA Adapter
api.upload_folder(
    repo_id=repo_id,
    repo_type="model",
    folder_path="/kaggle/working/medical-llm-qlora/outputs/clean_main_v1/training/adapter",
)

print("Upload complete:", repo_id)

In [ ]:
from huggingface_hub import snapshot_download
from kaggle_secrets import UserSecretsClient
from pathlib import Path

token = UserSecretsClient().get_secret("HF_TOKEN")

path = snapshot_download(
    repo_id="Lengbuding/llama3-medquad-qlora",
    token=token,
    local_dir="/kaggle/working/restored_adapter",
)

print("Downloaded to:", path)

for f in [
    "adapter_model.safetensors",
    "adapter_config.json",
    "tokenizer_config.json",
]:
    print(f, (Path(path) / f).exists())

In [ ]:
!python scripts/run_inference.py \
  --config configs/qlora_llama3_8b.yaml \
  --data outputs/clean_main_v1/data/evaluation_300.jsonl \
  --adapter /kaggle/working/restored_adapter \
  --output /kaggle/working/restored_adapter_test.jsonl \
  --limit 1

In [ ]:
import hashlib
from pathlib import Path

original = Path(
    "/kaggle/working/medical-llm-qlora/outputs/clean_main_v1/training/adapter"
)
restored = Path(
    "/kaggle/working/restored_adapter"
)

files = [
    "adapter_model.safetensors",
    "adapter_config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "chat_template.jinja",
]

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

all_same = True

for name in files:
    p1 = original / name
    p2 = restored / name

    if not p1.exists() or not p2.exists():
        print(f"{name}: 文件缺失")
        all_same = False
        continue

    same = sha256(p1) == sha256(p2)
    print(f"{name}: {same}")
    all_same &= same

print("\n核心文件完全一致:", all_same)

In [ ]:
!python scripts/run_inference.py \
  --config configs/qlora_llama3_8b.yaml \
  --data outputs/clean_main_v1/data/evaluation_300.jsonl \
  --adapter outputs/clean_main_v1/training/adapter \
  --output /kaggle/working/original_adapter_test.jsonl \
  --limit 1

In [ ]:
import json

with open("/kaggle/working/original_adapter_test.jsonl", encoding="utf-8") as f:
    original = json.loads(f.readline())

with open("/kaggle/working/restored_adapter_test.jsonl", encoding="utf-8") as f:
    restored = json.loads(f.readline())

print("sample_id 一致:", original["sample_id"] == restored["sample_id"])
print("prompt_hash 一致:", original["prompt_hash"] == restored["prompt_hash"])
print("generation_hash 一致:", original["generation_hash"] == restored["generation_hash"])
print("Base 输出一致:", original["base_prediction"] == restored["base_prediction"])
print("FT 输出一致:", original["ft_prediction"] == restored["ft_prediction"])

print("\n原始 FT 输出：")
print(original["ft_prediction"])

print("\nRestored FT 输出：")
print(restored["ft_prediction"])

## Rank pilot

Run the fixed rank 8/16 pilot and inspect the generated summary. Public result synchronization is handled outside the GPU notebook.


In [ ]:
!python scripts/run_ablation.py \
  --data-dir outputs/clean_main_v1/data \
  --output-dir outputs/ablation_rank

In [ ]:
import json

with open("outputs/ablation_rank/ablation_summary.json") as f:
    result = json.load(f)

print(json.dumps(result, indent=2))

## PubMedQA transfer check

Run the fixed 100-sample external capability check and inspect its summary.


In [ ]:
!python scripts/run_external_eval.py \
  --adapter outputs/clean_main_v1/training/adapter \
  --output-dir outputs/pubmedqa_external

In [ ]:
import json

with open("outputs/pubmedqa_external/pubmedqa_summary.json") as f:
    result = json.load(f)

print(json.dumps(result, indent=2))

## Repository synchronization

Do not store GitHub credentials or push commits from this notebook. Copy the small verified summaries into the local repository, run tests, review the diff, and push through the normal local Git workflow.
